# Federated Learning Simulation using DenseNet-121

This notebook implements a Federated Learning framework using a **DenseNet-121** architecture as the global model.

We conduct a comparative analysis by evaluating the model's performance under two distinct data distribution scenarios across simulated clients:
1. **Identically and Independently Distributed (IID) Data**
2. **Non-Identically and Independently Distributed (Non-IID) Data**

Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Imports

In [4]:
import os, gc, json, time, random, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from itertools import combinations

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

import matplotlib.pyplot as plt

Setup GPU

In [10]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
DEVICE

device(type='cpu')

Setting up Paths and Configurations

In [5]:
SEED = 42
BASE_PATH = Path('/content/drive/MyDrive/fyp')
IMAGE_DIR = BASE_PATH / 'Processed Dataset' / 'images'
SPLIT_DIR = BASE_PATH / 'results'  # Existing centralized CSVs, left untouched

OUTPUT_ROOT = BASE_PATH / 'Federated_Data_Final'
MODEL_ROOT = BASE_PATH / 'models' / 'federated'
RESULT_ROOT = BASE_PATH / 'results' / 'federated'
CLEAN_SPLIT_DIR = RESULT_ROOT / 'clean_splits'
for p in (OUTPUT_ROOT, MODEL_ROOT, RESULT_ROOT, CLEAN_SPLIT_DIR):
    p.mkdir(parents=True, exist_ok=True)

CFG = {
    'image_size': 224,
    'batch_size': 16,
    'rounds': 8,
    'local_epochs': 2,
    'head_only_rounds': 1,
    'lr_head': 2e-4,
    'lr_finetune': 1e-5,
    'weight_decay': 1e-4,
    'alpha': 0.5,
    'num_workers': 2,
    'num_classes': 4,
    'seed': SEED,
}
CLASS_NAMES = ['no_tumor','meningioma','glioma','pituitary']

assert IMAGE_DIR.is_dir(), f'Image folder missing: {IMAGE_DIR}'
for name in ('train','val','test'):
    assert (SPLIT_DIR / f'{name}.csv').is_file(), f'Missing {name}.csv in {SPLIT_DIR}'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print('Images:', IMAGE_DIR)
print('Original splits:', SPLIT_DIR)
print('New, cleaned evaluation splits:', CLEAN_SPLIT_DIR)
print('Hospital CSVs:', OUTPUT_ROOT)

Images: /content/drive/MyDrive/fyp/Processed Dataset/images
Original splits: /content/drive/MyDrive/fyp/results
New, cleaned evaluation splits: /content/drive/MyDrive/fyp/results/federated/clean_splits
Hospital CSVs: /content/drive/MyDrive/fyp/Federated_Data_Final


### Optimize Compute: Transfer Dataset to Local Storage
Copying the image files from Google Drive to the local SSD of the Colab runtime avoids bottlenecking model training with Drive I/O speeds.

In [9]:
import shutil

# Define local path
LOCAL_IMAGE_DIR = Path("/content/images")

# Create local directory if it doesn't exist
LOCAL_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("Copying images to local storage... This might take a few minutes depending on dataset size.")
# Copying all files from Google Drive to local disk
if IMAGE_DIR.exists():
    for img_path in tqdm(list(IMAGE_DIR.glob("*")), desc="Copying files"):
        if img_path.is_file():
            shutil.copy(img_path, LOCAL_IMAGE_DIR / img_path.name)

    # Update the IMAGE_DIR path for the training pipeline to use local SSD
    IMAGE_DIR = LOCAL_IMAGE_DIR
    print(f"Success! Images copied to local directory: {IMAGE_DIR}")
    print(f"Total local images: {len(list(IMAGE_DIR.glob('*')))}")
else:
    print(f"Source directory not found: {IMAGE_DIR}")

Copying images to local storage... This might take a few minutes depending on dataset size.


Copying files:   0%|          | 0/4478 [00:00<?, ?it/s]

Success! Images copied to local directory: /content/images
Total local images: 4478


Read and Validate the Train/Val/Test split // Pre-Training Safety Checks

In [15]:

# Load existing centralized splits
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "val.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")

splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}

required = {"image", "label", "patient_id"}

# Validate each dataset
for name, df in splits.items():

    assert required.issubset(df.columns), f"{name}: Missing columns"
    assert df[list(required)].notna().all().all(), f"{name}: Missing values"

    df["image"] = df["image"].astype(str).str.strip()
    df["patient_id"] = df["patient_id"].astype(str).str.strip()
    df["label"] = df["label"].astype(int)

    assert df["image"].ne("").all(), f"{name}: Blank filename"
    assert df["patient_id"].ne("").all(), f"{name}: Blank patient ID"
    assert df["label"].isin(range(4)).all(), f"{name}: Invalid labels"
    assert df["image"].is_unique, f"{name}: Duplicate filenames"

    # Create unique patient grouping
    source = (
        df["source"].fillna("unknown").astype(str)
        if "source" in df.columns else "unknown"
    )

    df["group_key"] = source + "::" + df["patient_id"]

    # Check image files exist
    missing = [
        img for img in df["image"]
        if not (IMAGE_DIR / img).is_file()
    ]

    assert not missing, f"{name}: Missing images {missing[:5]}"

# Check overlap between dataset splits


for (name1, df1), (name2, df2) in combinations(splits.items(), 2):

    assert set(df1["image"]).isdisjoint(df2["image"]), (
        f"Image overlap: {name1} and {name2}"
    )

    assert set(df1["group_key"]).isdisjoint(df2["group_key"]), (
        f"Patient overlap: {name1} and {name2}"
    )

# Display summary
for name, df in splits.items():
    print(f"{name}: {len(df)} images")

print("\nTraining class distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nAll validation checks passed!")


train: 3525 images
val: 501 images
test: 452 images

Training class distribution:
label
0    1131
1     486
2    1156
3     752
Name: count, dtype: int64

All validation checks passed!


Detect exact duplicate images and build uncontaminated evaluation subsets

In [17]:
RUN_EXACT_HASH_AUDIT = True  # Set True to run once; may be slow over Google Drive.
if RUN_EXACT_HASH_AUDIT:
    split_names = [('train',train_df), ('val',val_df), ('test',test_df)]
    sha_to_splits = {}
    for split_name, frame in split_names:
        for fname in tqdm(frame.image, desc=f'SHA256: {split_name}'):
            h = hashlib.sha256()
            with open(IMAGE_DIR / fname, 'rb') as f:
                for chunk in iter(lambda: f.read(1024*1024), b''):
                    h.update(chunk)
            sha_to_splits.setdefault(h.hexdigest(), set()).add(split_name)

    duplicated = [s for s in sha_to_splits.values() if len(s) > 1]
    print(f'{len(duplicated)} exact image-hash groups occur across splits.')
    if not duplicated:
        print('No byte-identical cross-split files found.')

SHA256: train:   0%|          | 0/3525 [00:00<?, ?it/s]

SHA256: val:   0%|          | 0/501 [00:00<?, ?it/s]

SHA256: test:   0%|          | 0/452 [00:00<?, ?it/s]

47 exact image-hash groups occur across splits.


### Evaluation Security Warning: Cross-Split Data Leakage

Our integrity audit identified **47 exact image-hash duplicates** shared across the train, validation, and test splits.

* **Why this occurs:** Overlaps of the same physical image under different filenames in the raw datasets.
* **The Impact:** While minor duplication might be tolerated during preliminary model selection experiments, it introduces **data leakage** into our final evaluation splits. This artificially inflates performance metrics.
* **Our Solution:** To guarantee a strict and scientifically sound evaluation of our Federated Learning model, we must remove these duplicates from the validation and test splits. This ensures our final evaluation metrics are entirely untainted.

Detect exact duplicate images and build uncontaminated evaluation subsets

In [20]:

# 1. Calculate SHA256 hash for every image
def image_sha256(filename):
    return hashlib.sha256(
        (IMAGE_DIR / filename).read_bytes()
    ).hexdigest()

for name, df in [("train", train_df),("val", val_df),("test", test_df)]:

    df["sha256"] = [
        image_sha256(img)
        for img in tqdm(df["image"], desc=f"SHA256: {name}")
    ]


# 2. Check identical images have the same label
all_data = pd.concat([train_df, val_df, test_df])

conflicts = all_data.groupby("sha256")["label"].nunique()

assert (conflicts <= 1).all(), "Identical images have conflicting labels!"


# 3. Remember original dataset information
original_sizes = {
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df)
}

train_hashes = set(train_df["sha256"])
val_hashes = set(val_df["sha256"])

train_patients = set(train_df["group_key"])
val_patients = set(val_df["group_key"])


# 4. Remove duplicate image content from evaluation datasets
val_df = val_df[
    ~val_df["sha256"].isin(train_hashes)
].drop_duplicates("sha256").copy()

test_df = test_df[
    ~test_df["sha256"].isin(train_hashes | val_hashes)
].drop_duplicates("sha256").copy()


# 5. Remove known patient overlap
val_df = val_df[
    ~val_df["group_key"].isin(train_patients)
].copy()

test_df = test_df[
    ~test_df["group_key"].isin(train_patients | val_patients)
].copy()


# 6. Verify no leakage remains
for a, b in [
    (train_df, val_df),
    (train_df, test_df),
    (val_df, test_df)
]:
    assert set(a["sha256"]).isdisjoint(set(b["sha256"]))
    assert set(a["group_key"]).isdisjoint(set(b["group_key"]))

assert len(val_df) > 0 and len(test_df) > 0, "Empty evaluation split!"

assert val_df["label"].nunique() == 4
assert test_df["label"].nunique() == 4


# 7. Save cleaned CSV files
for name, df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df)
]:
    df.to_csv(CLEAN_SPLIT_DIR / f"{name}.csv", index=False)

    print(
        f"{name}: {original_sizes[name]} -> {len(df)} images"
    )


print("\nValidation class distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

print("\nDataset cleaning completed successfully!")


SHA256: train:   0%|          | 0/3525 [00:00<?, ?it/s]

SHA256: val:   0%|          | 0/478 [00:00<?, ?it/s]

SHA256: test:   0%|          | 0/422 [00:00<?, ?it/s]

train: 3525 -> 3525 images
val: 478 -> 478 images
test: 422 -> 422 images

Validation class distribution:
label
0    119
1    106
2    178
3     75
Name: count, dtype: int64

Test class distribution:
label
0    111
1    116
2     92
3    103
Name: count, dtype: int64

Dataset cleaning completed successfully!


In [21]:

# Four simulated hospitals
HOSPITALS = ["Hospital_A", "Hospital_B", "Hospital_C", "Hospital_D"]
NUM_CLIENTS = len(HOSPITALS)

# Initially, each patient is a separate group
parent = {key: key for key in train_df["group_key"].unique()}

# Find the main group of a patient
def find(key):
    while parent[key] != key:
        parent[key] = parent[parent[key]]
        key = parent[key]
    return key

# Merge patients who have identical MRI images
for _, group in train_df.groupby("sha256"):

    patients = group["group_key"].unique()

    for patient in patients[1:]:
        a = find(patients[0])
        b = find(patient)

        if a != b:
            parent[max(a, b)] = min(a, b)

# Update patient groups
train_df["group_key"] = train_df["group_key"].map(find)

# Ensure every combined group belongs to one class
class_counts = train_df.groupby("group_key")["label"].nunique()

assert (class_counts <= 1).all(), (
    "Some patient groups contain conflicting labels!"
)

# Create patient summary for hospital distribution
patient_table = (
    train_df.groupby("group_key", as_index=False)
    .agg(
        label=("label", "first"),
        n_images=("image", "size")
    )
)

# Save updated grouping
train_df.to_csv(CLEAN_SPLIT_DIR / "train.csv", index=False)

# Display results
print("Total training images:", len(train_df))
print("Total patient groups:", len(patient_table))

print("\nClass distribution:")
print(
    patient_table.groupby("label").agg(
        groups=("group_key", "size"),
        images=("n_images", "sum")
    )
)


Total training images: 3525
Total patient groups: 1222

Class distribution:
       groups  images
label                
0        1036    1131
1          63     486
2          73    1156
3          50     752


Creating IID and Non-IID Distributions

In [22]:

# 1. Assign patient groups to hospitals
def assign_groups(non_iid=False):

    rng = np.random.default_rng(SEED)
    assignments = {}

    class_counts = np.zeros((NUM_CLIENTS, 4), dtype=int)
    total_counts = np.zeros(NUM_CLIENTS, dtype=int)

    for label in range(4):

        groups = patient_table[patient_table["label"] == label].copy()

        # Random ordering for equal-sized groups
        groups["_tie"] = rng.random(len(groups))
        groups = groups.sort_values(
            ["n_images", "_tie"],
            ascending=[False, True]
        )

        if non_iid:
            targets = rng.dirichlet(
                [CFG["alpha"]] * NUM_CLIENTS
            ) * groups["n_images"].sum()

            allocated = np.zeros(NUM_CLIENTS)

        for row in groups.itertuples(index=False):

            size = int(row.n_images)

            if non_iid:
                # Assign according to Dirichlet target
                client = int(np.argmax(targets - allocated))
                allocated[client] += size

            else:
                # Balance class image counts across hospitals
                scores = class_counts[:, label] * 100000 + total_counts
                client = int(np.argmin(scores))

                class_counts[client, label] += size
                total_counts[client] += size

            assignments[row.group_key] = client

    return assignments


# 2. Create hospital DataFrames
def build_clients(assignments):

    data = train_df.copy()
    data["client_id"] = data["group_key"].map(assignments)

    assert data["client_id"].notna().all()

    return {
        name: data[data["client_id"] == i].copy().reset_index(drop=True)
        for i, name in enumerate(HOSPITALS)
    }


# 3. Validate hospital partitions
def validate_clients(clients):

    combined = pd.concat(clients.values(), ignore_index=True)

    # All training images are assigned exactly once
    assert len(combined) == len(train_df)
    assert combined["image"].is_unique
    assert set(combined["image"]) == set(train_df["image"])

    # Each hospital must have enough images
    assert all(len(df) >= 16 for df in clients.values())

    # Patients and identical images cannot cross hospitals
    assert (combined.groupby("group_key")["client_id"].nunique() == 1).all()
    assert (combined.groupby("sha256")["client_id"].nunique() == 1).all()

    # No validation/test images in training
    eval_images = set(val_df["image"]) | set(test_df["image"])
    eval_hashes = set(val_df["sha256"]) | set(test_df["sha256"])

    assert set(combined["image"]).isdisjoint(eval_images)
    assert set(combined["sha256"]).isdisjoint(eval_hashes)

    print("Validation passed! All hospital partitions are safe.")


# 4. Generate IID and Non-IID clients
IID_ASSIGN = assign_groups(non_iid=False)
NONIID_ASSIGN = assign_groups(non_iid=True)

IID_CLIENTS = build_clients(IID_ASSIGN)
NONIID_CLIENTS = build_clients(NONIID_ASSIGN)

validate_clients(IID_CLIENTS)
validate_clients(NONIID_CLIENTS)


# 5. Show hospital distributions
for scenario, clients in [
    ("IID", IID_CLIENTS),
    ("NON_IID_Dirichlet", NONIID_CLIENTS)
]:

    print(f"\n{scenario} Distribution")

    for name, df in clients.items():
        print(
            f"{name}: {len(df)} images | "
            f"{df['group_key'].nunique()} groups | "
            f"Classes: {df['label'].value_counts().sort_index().to_dict()}"
        )


Validation passed! All hospital partitions are safe.
Validation passed! All hospital partitions are safe.

IID Distribution
Hospital_A: 880 images | 305 groups | Classes: {0: 283, 1: 122, 2: 289, 3: 186}
Hospital_B: 885 images | 305 groups | Classes: {0: 283, 1: 121, 2: 289, 3: 192}
Hospital_C: 882 images | 307 groups | Classes: {0: 283, 1: 121, 2: 290, 3: 188}
Hospital_D: 878 images | 305 groups | Classes: {0: 282, 1: 122, 2: 288, 3: 186}

NON_IID_Dirichlet Distribution
Hospital_A: 1007 images | 234 groups | Classes: {0: 168, 1: 29, 2: 76, 3: 734}
Hospital_B: 848 images | 318 groups | Classes: {0: 277, 1: 33, 2: 535, 3: 3}
Hospital_C: 969 images | 85 groups | Classes: {0: 12, 1: 423, 2: 527, 3: 7}
Hospital_D: 701 images | 585 groups | Classes: {0: 674, 1: 1, 2: 18, 3: 8}


In [23]:

# 1. Save hospital CSV files
scenarios = {
    "IID": IID_CLIENTS,
    "NON_IID_Dirichlet": NONIID_CLIENTS
}

for scenario, hospitals in scenarios.items():
    for hospital, df in hospitals.items():

        file_path = OUTPUT_ROOT / scenario / hospital / "metadata.csv"
        file_path.parent.mkdir(parents=True, exist_ok=True)

        df.drop(columns=["group_key"], errors="ignore").to_csv(
            file_path, index=False
        )

        print("Saved:", file_path)


# 2. Save partition information
manifest = {
    "seed": SEED,
    "alpha": CFG["alpha"],
    "train_count": len(train_df),
    "validation_count": len(val_df),
    "test_count": len(test_df),
    "train_image_names_sha256": hashlib.sha256(
        "\n".join(sorted(train_df["image"])).encode()
    ).hexdigest(),
    "num_clients": NUM_CLIENTS,
    "grouping": "source::patient_id",
    "note": "No-tumor missing real patient IDs: synthetic groups are not verified distinct patients."
}

with open(OUTPUT_ROOT / "partition_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nAll hospital datasets and partition information saved!")


Saved: /content/drive/MyDrive/fyp/Federated_Data_Final/IID/Hospital_A/metadata.csv
Saved: /content/drive/MyDrive/fyp/Federated_Data_Final/IID/Hospital_B/metadata.csv
Saved: /content/drive/MyDrive/fyp/Federated_Data_Final/IID/Hospital_C/metadata.csv
Saved: /content/drive/MyDrive/fyp/Federated_Data_Final/IID/Hospital_D/metadata.csv
Saved: /content/drive/MyDrive/fyp/Federated_Data_Final/NON_IID_Dirichlet/Hospital_A/metadata.csv
Saved: /content/drive/MyDrive/fyp/Federated_Data_Final/NON_IID_Dirichlet/Hospital_B/metadata.csv
Saved: /content/drive/MyDrive/fyp/Federated_Data_Final/NON_IID_Dirichlet/Hospital_C/metadata.csv
Saved: /content/drive/MyDrive/fyp/Federated_Data_Final/NON_IID_Dirichlet/Hospital_D/metadata.csv

All hospital datasets and partition information saved!


Defining DataSet and Loaders

In [24]:
MEAN=[0.485,0.456,0.406]
STD=[0.229,0.224,0.225]
train_transform = transforms.Compose([
    transforms.Resize((CFG['image_size'],CFG['image_size'])),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((CFG['image_size'],CFG['image_size'])),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

class BrainMRIDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, index):
        row = self.df.iloc[index]
        with Image.open(self.image_dir / row.image) as im:
            image = im.convert('RGB')
        return self.transform(image), int(row.label)

def make_loader(frame, train=False):
    return DataLoader(
        BrainMRIDataset(frame, IMAGE_DIR, train_transform if train else eval_transform),
        batch_size=CFG['batch_size'], shuffle=train,
        num_workers=CFG['num_workers'], pin_memory=True,
    )

VAL_LOADER = make_loader(val_df, train=False)
TEST_LOADER = make_loader(test_df, train=False)
# Only use TEST_LOADER in the final evaluation cell, NOT during training/selection.

DenseNet121 model factory + initial global state

In [8]:
def make_densenet(pretrained=False):
    weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
    model = models.densenet121(weights=weights)
    model.classifier = nn.Linear(model.classifier.in_features, CFG['num_classes'])
    return model

seed_everything(SEED)
init_model = make_densenet(pretrained=True)  # one-time ImageNet weight download
INITIAL_STATE = {k:v.detach().cpu().clone() for k,v in init_model.state_dict().items()}
del init_model
gc.collect()
print('Initialized DenseNet121:', len(INITIAL_STATE), 'state entries')

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 118MB/s]


Initialized DenseNet121: 727 state entries


Class-weighted local training

In [27]:
counts = train_df.label.value_counts().reindex(range(4),fill_value=0).to_numpy()
assert (counts > 0).all(), 'Expected all four classes in the training dataset'
weight_values = (len(train_df)/(4*counts)).astype(np.float32)
CLASS_WEIGHTS = torch.tensor(weight_values, dtype=torch.float32, device=DEVICE)
print('Class weights:', CLASS_WEIGHTS.tolist())


def set_trainable_params(model, head_only):
    for p in model.parameters(): p.requires_grad = not head_only
    for p in model.classifier.parameters(): p.requires_grad = True


def train_local_model(model, loader, round_id, hospital):
    head_only = round_id <= CFG['head_only_rounds']
    set_trainable_params(model, head_only)
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=CFG['lr_head'] if head_only else CFG['lr_finetune'],
        weight_decay=CFG['weight_decay']
    )
    loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    steps, total_loss, num_right, num_seen = 0,0.0,0,0
    for ep in range(CFG['local_epochs']):
        model.train()
        if head_only: model.features.eval() # prevent BN updates while freezing backbone
        for images, labels in tqdm(loader, desc=f'Round {round_id} {hospital} ep {ep+1}', leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = loss_fn(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            batch_n = labels.size(0)
            num_seen += batch_n
            total_loss += loss.item()*batch_n
            num_right += (logits.argmax(1)==labels).sum().item()
            steps += 1
    return {'train_loss':total_loss/num_seen, 'train_acc':num_right/num_seen,
            'train_examples_seen':num_seen}

Class weights: [0.779177725315094, 1.8132716417312622, 0.7623270153999329, 1.171875]


FedAvg aggregation + validation evaluator

In [28]:
def fedavg(client_states, sizes):
    assert len(client_states)==len(sizes) and min(sizes)>0
    total = float(sum(sizes))
    largest_idx = int(np.argmax(sizes))
    out = {}
    for key in client_states[0]:
        sample = client_states[0][key]
        if sample.is_floating_point():
            # Float32 accumulator even when training used AMP.
            acc = torch.zeros_like(sample, dtype=torch.float32, device='cpu')
            for state,n in zip(client_states, sizes):
                acc.add_(state[key].float(), alpha=float(n)/total)
            out[key] = acc.to(dtype=sample.dtype)
        else:
            out[key] = client_states[largest_idx][key].clone()
    return out

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    true, pred = [], []
    for x,y in loader:
        logits = model(x.to(DEVICE, non_blocking=True))
        true.extend(y.numpy().tolist())
        pred.extend(logits.argmax(1).cpu().numpy().tolist())
    return {
        'accuracy':accuracy_score(true,pred),
        'precision_macro':precision_score(true,pred,labels=list(range(4)),average='macro',zero_division=0),
        'recall_macro':recall_score(true,pred,labels=list(range(4)),average='macro',zero_division=0),
        'f1_macro':f1_score(true,pred,labels=list(range(4)),average='macro',zero_division=0),
    }, true, pred

In [29]:
def run_federated_experiment(scenario, clients):
    print('\n'+'='*66+'\n',scenario, 'FEDERATED TRAINING\n'+'='*66)
    model_dir = MODEL_ROOT / scenario
    result_dir = RESULT_ROOT / scenario
    model_dir.mkdir(parents=True,exist_ok=True)
    result_dir.mkdir(parents=True,exist_ok=True)

    global_state = {k:v.clone() for k,v in INITIAL_STATE.items()}
    best_f1, best_round = -1.0, 0
    logs=[]
    all_local_metrics=[]

    # Local loaders are based ONLY on patient-safe train-only hospital CSVs.
    loaders = {name:make_loader(part, train=True) for name,part in clients.items()}
    sizes = {name:len(part) for name,part in clients.items()}
    for round_id in range(1, CFG['rounds']+1):
        start = time.monotonic()
        client_states=[]
        client_sizes=[]
        local_metrics=[]
        for hospital, loader in loaders.items():
            model = make_densenet(pretrained=False).to(DEVICE)
            model.load_state_dict(global_state, strict=True)  # SAME global start for each hospital!
            result = train_local_model(model,loader,round_id,hospital)
            state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            client_states.append(state)
            client_sizes.append(sizes[hospital])
            local_metrics.append({'round':round_id,'hospital':hospital,
                                  'samples':sizes[hospital],**result})
            del model
            torch.cuda.empty_cache()
            print(f'  {hospital}: samples={sizes[hospital]}, local acc={result["train_acc"]:.3f}')

        global_state = fedavg(client_states, client_sizes)
        del client_states
        gc.collect()

        model = make_densenet(pretrained=False).to(DEVICE)
        model.load_state_dict(global_state, strict=True)
        val_metrics,_,_ = evaluate(model, VAL_LOADER)
        del model
        torch.cuda.empty_cache()
        elapsed = round(time.monotonic()-start,1)
        record = {'round':round_id, 'scenario':scenario,
                  'seconds':elapsed, **{f'val_{k}':v for k,v in val_metrics.items()}}
        logs.append(record)
        pd.DataFrame(logs).to_csv(result_dir/'round_history.csv', index=False)
        all_local_metrics.extend(local_metrics)
        pd.DataFrame(all_local_metrics).to_csv(result_dir/'local_history.csv', index=False)
        print(f'ROUND {round_id}: val acc={val_metrics["accuracy"]:.4f} '
              f'val macro-F1={val_metrics["f1_macro"]:.4f} time={elapsed}s')
        if val_metrics['f1_macro'] > best_f1:
            best_f1 = val_metrics['f1_macro']; best_round = round_id
            torch.save(global_state, model_dir/'best_global_weights.pth')
            print('  Saved new best global checkpoint.')
        torch.save(global_state, model_dir/'latest_global_weights.pth')

    summary = {'scenario':scenario,'best_round':best_round,'best_val_macro_f1':best_f1,
               'rounds_run':CFG['rounds'],'n_clients':len(clients),
               'client_sample_counts':sizes, 'architecture':'densenet121'}
    (result_dir/'summary.json').write_text(json.dumps(summary,indent=2))
    (model_dir/'model_config.json').write_text(json.dumps({
        'architecture':'densenet121','num_classes':4,'class_names':CLASS_NAMES,
        'image_size':CFG['image_size'],'rgb':True,
        'imagenet_mean':MEAN, 'imagenet_std':STD,
        'best_weights_file':'best_global_weights.pth',
        'selection_metric':'val_macro_f1', 'scenario':scenario
    },indent=2))
    return summary

Start Federated Training

In [31]:
# Toggle one scenario at a time if compute is limited.
RUN_IID = False
RUN_NONIID = True

summaries = []
if RUN_IID:
    summaries.append(run_federated_experiment('IID', IID_CLIENTS))
if RUN_NONIID:
    summaries.append(run_federated_experiment('NON_IID_Dirichlet', NONIID_CLIENTS))

if summaries:
    display(pd.DataFrame(summaries)[['scenario','best_round','best_val_macro_f1','rounds_run']])


 NON_IID_Dirichlet FEDERATED TRAINING


Round 1 Hospital_A ep 1:   0%|          | 0/63 [00:00<?, ?it/s]

Round 1 Hospital_A ep 2:   0%|          | 0/63 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b9faabec2c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b9faabec2c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

  Hospital_A: samples=1007, local acc=0.764


...

  Hospital_C: samples=969, local acc=0.927


Round 8 Hospital_D ep 1:   0%|          | 0/44 [00:00<?, ?it/s]

Round 8 Hospital_D ep 2:   0%|          | 0/44 [00:00<?, ?it/s]

  Hospital_D: samples=701, local acc=0.797
ROUND 8: val acc=0.8536 val macro-F1=0.8392 time=145.8s
  Saved new best global checkpoint.


,scenario,best_round,best_val_macro_f1,rounds_run
0,NON_IID_Dirichlet,8,0.839185,8


Evaluating IID and NON-IID Models on Test Set

In [11]:

# Evaluate both federated models on the cleaned test set

results = []

for scenario in ["IID", "NON_IID_Dirichlet"]:

    # Load best global checkpoint
    checkpoint = MODEL_ROOT / scenario / "best_global_weights.pth"

    assert checkpoint.exists(), f"Missing checkpoint: {checkpoint}"

    model = make_densenet(pretrained=False).to(DEVICE)
    model.load_state_dict(
        torch.load(checkpoint, map_location="cpu", weights_only=True)
    )

    # Test evaluation
    metrics, y_true, y_pred = evaluate(model, TEST_LOADER)

    print(f"\n{'=' * 50}")
    print(f"{scenario} - Test Results ({len(test_df)} images)")
    print(f"{'=' * 50}")

    report = classification_report(
        y_true, y_pred,
        labels=range(4),
        target_names=CLASS_NAMES,
        zero_division=0
    )

    print(report)

    # Save classification report
    folder = RESULT_ROOT / scenario
    folder.mkdir(parents=True, exist_ok=True)

    (folder / "test_classification_report.txt").write_text(report)

    # Save confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=range(4))

    pd.DataFrame(
        cm, index=CLASS_NAMES, columns=CLASS_NAMES
    ).to_csv(folder / "test_confusion_matrix.csv")

    # Store metrics
    results.append({
        "model": scenario,
        "test_images": len(y_true),
        **metrics
    })

    del model
    torch.cuda.empty_cache()


# Compare federated models
federated_results = pd.DataFrame(results)

display(federated_results.round(4))

federated_results.to_csv(
    RESULT_ROOT / "federated_test_comparison.csv",
    index=False
)

print("Federated test evaluation completed!")


Final Evaluation of Centralized DenseNet on Cleaned Test set

In [34]:

# Path to the original centralized DenseNet121 checkpoint
CENTRAL_WEIGHTS = BASE_PATH / "models" / "densenet121_best.pth"

assert CENTRAL_WEIGHTS.exists(), f"Model not found: {CENTRAL_WEIGHTS}"

# Load centralized model
central_model = make_densenet(pretrained=False).to(DEVICE)

central_model.load_state_dict(
    torch.load(CENTRAL_WEIGHTS, map_location="cpu", weights_only=True)
)

# Evaluate on the SAME cleaned test set
central_metrics, y_true, y_pred = evaluate(
    central_model, TEST_LOADER
)

print("\nCentralized DenseNet121 - Clean Test Results")
print("Test images:", len(y_true))

report = classification_report(
    y_true, y_pred,
    labels=range(4),
    target_names=CLASS_NAMES,
    zero_division=0
)

print(report)

# Save classification report
(RESULT_ROOT / "centralized_test_report.txt").write_text(report)

# Save centralized model metrics
with open(RESULT_ROOT / "centralized_clean_test_metrics.json", "w") as f:
    json.dump(central_metrics, f, indent=2)

# Final comparison of all 3 models
central_result = pd.DataFrame([{
    "model": "Centralized",
    "test_images": len(y_true),
    **central_metrics
}])

final_comparison = pd.concat(
    [central_result, federated_results],
    ignore_index=True
)

print("\nFinal Model Comparison:")
display(final_comparison.round(4))

final_comparison.to_csv(
    RESULT_ROOT / "final_clean_test_comparison.csv",
    index=False
)

del central_model
torch.cuda.empty_cache()

print("Final evaluation completed!")



Centralized DenseNet121 - Clean Test Results
Test images: 422
              precision    recall  f1-score   support

    no_tumor       1.00      1.00      1.00       111
  meningioma       0.97      0.98      0.97       116
      glioma       0.98      0.97      0.97        92
   pituitary       1.00      0.99      1.00       103

    accuracy                           0.99       422
   macro avg       0.99      0.99      0.99       422
weighted avg       0.99      0.99      0.99       422


Final Model Comparison:


,model,test_images,accuracy,precision_macro,recall_macro,f1_macro
0,Centralized,422,0.9858,0.9860,0.9851,0.9855
1,IID,422,0.9408,0.9444,0.9393,0.9411
2,NON_IID_Dirichlet,422,0.8294,0.8451,0.8356,0.8311


Final evaluation completed!
